In [ ]:
import torch
import gzip
import logging
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import einops
import io
import base64
from PIL import Image
from os.path import join as opj
import random
from functools import partial
import math
from torchvision import transforms

from torchvision.transforms.functional import adjust_contrast, adjust_brightness

In [ ]:
class GaussianBlur(transforms.RandomApply):
    """
    Apply Gaussian Blur to the PIL image.
    """

    def __init__(self,
                 *,
                 p: float = 0.5,
                 radius_min: float = 0.1,
                 radius_max: float = 2.0):
        # NOTE: torchvision is applying 1 - probability to return the original image
        keep_p = 1 - p
        transform = transforms.GaussianBlur(kernel_size=9,
                                            sigma=(radius_min, radius_max))
        super().__init__(transforms=[transform], p=keep_p)

In [ ]:
class RandomRangeCrop(torch.nn.Module):
    """
    Always crop a square patch of side `size` from a fixed-size image (HxH),
    where both the row and column start are sampled independently from
    the same [min_start, max_start] interval (inclusive).
    Designed to crop 64x64 to 48x48. top left range would be 0,0 - 16,16. shift factor should be between [0-7].
    0 would be most aggressive.
    Args:
        size (int): side length of the square crop.
        shift_factor (int): how much shift to exclude
        img_size (int): the constant height/width H of all input images.
    """
    def __init__(self, size: int, shift_factor: int):
        super().__init__()
        self.size = size
        mn, mx = shift_factor, 16-shift_factor

        self.min_start = mn
        self.max_start = mx

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        """
        Args:
            img (Tensor): shape (C, H, H)

        Returns:
            Tensor: cropped image of shape (C, size, size)
        """
        # draw row and column starts independently from [min_start, max_start]
        i = torch.randint(self.min_start, self.max_start + 1, ()).item()
        j = torch.randint(self.min_start, self.max_start + 1, ()).item()

        return img[:, i : i + self.size, j : j + self.size]

In [ ]:
class GaussianNoise(torch.nn.Module):
    """object to add guassian noise to images."""

    def __init__(self, min_var: float = 0.01, max_var: float = 0.1):
        super().__init__()
        self.min_var = min_var
        self.max_var = max_var

    def forward(self, tensor):
        min_, max_ = tensor.min(), tensor.max()
        var = random.uniform(self.min_var, self.max_var)
        noisy = tensor + torch.randn(tensor.size()) * var
        return torch.clamp(noisy, min=min_, max=max_)

In [ ]:
def srh_to_uint8_func(x):
    return (adjust_brightness(adjust_contrast(x, 2), 3) * 255).to(torch.uint8) # (x-x.min()) / (x.max()-x.min())

In [ ]:
class MaskingGenerator:
    def __init__(
        self,
        input_size,
        num_masking_patches=None,
        min_num_patches=4,
        max_num_patches=None,
        min_aspect=0.3,
        max_aspect=None,
    ):
        if not isinstance(input_size, tuple):
            input_size = (input_size,) * 2
        self.height, self.width = input_size

        self.num_patches = self.height * self.width
        self.num_masking_patches = num_masking_patches

        self.min_num_patches = min_num_patches
        self.max_num_patches = num_masking_patches if max_num_patches is None else max_num_patches

        max_aspect = max_aspect or 1 / min_aspect
        self.log_aspect_ratio = (math.log(min_aspect), math.log(max_aspect))

    def __repr__(self):
        repr_str = "Generator(%d, %d -> [%d ~ %d], max = %d, %.3f ~ %.3f)" % (
            self.height,
            self.width,
            self.min_num_patches,
            self.max_num_patches,
            self.num_masking_patches,
            self.log_aspect_ratio[0],
            self.log_aspect_ratio[1],
        )
        return repr_str

    def get_shape(self):
        return self.height, self.width

    def _mask(self, mask, max_mask_patches):
        delta = 0
        for _ in range(10):
            target_area = random.uniform(self.min_num_patches, max_mask_patches)
            aspect_ratio = math.exp(random.uniform(*self.log_aspect_ratio))
            h = int(round(math.sqrt(target_area * aspect_ratio)))
            w = int(round(math.sqrt(target_area / aspect_ratio)))
            if w < self.width and h < self.height:
                top = random.randint(0, self.height - h)
                left = random.randint(0, self.width - w)

                num_masked = mask[top : top + h, left : left + w].sum()
                # Overlap
                if 0 < h * w - num_masked <= max_mask_patches:
                    for i in range(top, top + h):
                        for j in range(left, left + w):
                            if mask[i, j] == 0:
                                mask[i, j] = 1
                                delta += 1

                if delta > 0:
                    break
        return delta

    def __call__(self, num_masking_patches=0):
        mask = np.zeros(shape=self.get_shape(), dtype=bool)
        mask_count = 0
        while mask_count < num_masking_patches:
            max_mask_patches = num_masking_patches - mask_count
            max_mask_patches = min(max_mask_patches, self.max_num_patches)

            delta = self._mask(mask, max_mask_patches)
            if delta == 0:
                break
            else:
                mask_count += delta

        return mask

In [ ]:
class OuterBiasedMasker():

    def __init__(self,
                 mask_size: int,
                 token_size: int,
                 dist_power: float = 2.0,
                 sharpness_range=[1, 5],
                 n_mask_token_perct_range=[.5, .75],
                 device='cpu',
                 dtype=torch.float32):
        """
        Args:
            mask_size (int): size of the square mask (mask_size x mask_size).
            power (float): strength of the bias toward outer elements.
            device (str): device to use for tensors ('cpu' or 'cuda').
            dtype (torch.dtype): dtype for probability tensor.
        """
        self.mask_size = mask_size
        self.token_size = token_size
        self.device = device
        self.dtype = dtype
        self.sharpness_range = sharpness_range
        self.n_mask_token_min = int(
            round(n_mask_token_perct_range[0] * mask_size * mask_size))
        self.n_mask_token_max = int(
            round(n_mask_token_perct_range[1] * mask_size * mask_size))

        # Precompute distance
        x = torch.arange(mask_size, device=device, dtype=dtype)
        y = torch.arange(mask_size, device=device, dtype=dtype)
        yy, xx = torch.meshgrid(x, y, indexing='ij')

        cx, cy = (mask_size - 1) / 2, (mask_size - 1) / 2
        dist2 = (xx - cx)**dist_power + (yy - cy)**dist_power
        self.norm_dist = dist2 / dist2.max()  # normalize to [0, 1]
        self.upsample_kernel = torch.ones((token_size, token_size),
                                          dtype=torch.uint8,
                                          device=device)

    def __call__(self, return_flat_mask=False) -> torch.Tensor:
        """
        Sample a k x k binary mask with num_masked entries = 1, biased toward outer region.

        Args:
            num_masked (int): number of tokens to mask.

        Returns:
            torch.Tensor: mask of shape (k, k), dtype=torch.uint8.
        """
        num_masked = random.randint(self.n_mask_token_min,
                                    self.n_mask_token_max)
        sharpness = random.uniform(*self.sharpness_range)
        prob = self.norm_dist.pow(sharpness)
        prob = (prob / prob.sum()).flatten()
        flat_mask = torch.zeros(self.mask_size * self.mask_size,
                                dtype=torch.uint8,
                                device=self.device)
        idx = torch.multinomial(prob,
                                num_samples=num_masked,
                                replacement=False)
        flat_mask[idx] = 1
        base_mask = flat_mask.view(self.mask_size, self.mask_size)
        upsampled_full_mask = torch.kron(base_mask, self.upsample_kernel)
        if return_flat_mask:
            return flat_mask
        else:
            return upsampled_full_mask



In [ ]:
class CellCropNoAugmentationDINO(object):

    def __init__(
        self,
        global_crops_scale = [.75, 1.0],
        local_crops_scale = [0.05, 0.32],
        local_crops_number=8,
        global_crops_size=48,
        local_crops_size=20,
    ):


        self.local_crops_number = local_crops_number

        # random resized crop and flip
        geometric_augmentation_global = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomResizedCrop(
                    global_crops_size, scale=global_crops_scale, interpolation=transforms.InterpolationMode.BICUBIC
                ),
        ])
        geometric_augmentation_local = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomResizedCrop(
                    local_crops_size, scale=local_crops_scale, interpolation=transforms.InterpolationMode.BICUBIC
                ),
        ])

        self.global_transfo = transforms.Compose(
            [geometric_augmentation_global])
        self.local_transfo = transforms.Compose(
            [geometric_augmentation_local])
        self.dropna = partial(torch.nan_to_num, nan=1.0e-6, posinf=1, neginf=1.0e-6)


    def __call__(self, image):
        output = {}

        global_crops = [self.dropna(self.global_transfo(image))
                        for _ in range(2)]
        local_crops = [self.dropna(self.local_transfo(image))
                        for _ in range(self.local_crops_number)]

        output["local_crops"] = local_crops
        output["global_crops"] = global_crops
        output["global_crops_teacher"] = global_crops
        output["offsets"] = ()

        return output

In [ ]:
class CellAugmentationDINO(object):

    def __init__(
        self,
        local_crops_number=2,
        crops_size=48,
    ):
        self.local_crops_number = local_crops_number
        self.crops_size = crops_size

        logging.info("###################################")
        logging.info("Using data augmentation parameters:")
        logging.info(f"local_crops_number: {local_crops_number}")
        logging.info(f"crops_size: {crops_size}")
        logging.info("###################################")

        # random resized crop and flip
        geometric_augmentation_global = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            RandomRangeCrop(size=crops_size, shift_factor=4)
        ])


        # color distorsions / blurring
        color_jittering = transforms.Compose([
            transforms.RandomApply(
                [
                    transforms.ColorJitter(
                        brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
                ],
                p=0.8,
            ),
            transforms.RandomApply(
                [GaussianNoise(min_var=0, max_var=0.05)],p=0.5),
            GaussianBlur(p=0.5, radius_max=1),
            transforms.RandomSolarize(threshold=0.5, p=.2),
            transforms.RandomGrayscale(p=0.2),
        ])


        self.global_transfo = transforms.Compose(
            [geometric_augmentation_global])#, color_jittering])
        self.local_transfo = transforms.Compose(
            [geometric_augmentation_global])
        self.dropna = partial(torch.nan_to_num, nan=1.0e-6, posinf=1, neginf=1.0e-6)


    def __call__(self, image):
        output = {}

        global_crops = [self.dropna(self.global_transfo(image))
                        for _ in range(2)]
        local_crops = [self.dropna(self.local_transfo(image))
                        for _ in range(self.local_crops_number)]

        output["local_crops"] = local_crops
        output["global_crops"] = global_crops
        output["global_crops_teacher"] = global_crops
        output["offsets"] = ()

        return output

In [ ]:
class SRHInstanceNorm(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.instance_norm = torch.nn.InstanceNorm2d(num_features=3)
        self.mean = torch.tensor([0.0872, 0.1546, 0.1604]).unsqueeze(-1).unsqueeze(-1)
        self.std = torch.tensor([0.0444, 0.0939, 0.0560]).unsqueeze(-1).unsqueeze(-1)

    def __call__(self, image):
        return (self.instance_norm(image) * self.std + self.mean).clamp(0, 1)




In [ ]:
def create_mask(B,N, mask_ratio_tuple, mask_probability, mask_generator):
    n_samples_masked = int(B * mask_probability)
    probs = torch.linspace(*mask_ratio_tuple, n_samples_masked + 1)
    upperbound = 0
    masks_list = []
    for i in range(0, n_samples_masked):
        prob_min = probs[i]
        prob_max = probs[i + 1]
        masks_list.append(torch.BoolTensor(mask_generator(int(N * random.uniform(prob_min, prob_max)))))
        upperbound += int(N * prob_max)
    for i in range(n_samples_masked, B):
        masks_list.append(torch.BoolTensor(mask_generator(0)))

    random.shuffle(masks_list)

    collated_masks = torch.stack(masks_list).flatten(1)
    mask_indices_list = collated_masks.flatten().nonzero().flatten()

    masks_weight = (1 / collated_masks.sum(-1).clamp(min=1.0)).unsqueeze(-1).expand_as(collated_masks)[collated_masks]

    return {
        "collated_masks": collated_masks,
        "mask_indices_list": mask_indices_list,
        "masks_weight": masks_weight,
        "upperbound": upperbound,
        "n_masked_patches": torch.full((1,), fill_value=mask_indices_list.shape[0], dtype=torch.long),
    }

In [ ]:
def create_mask_with_bgswap_mask(B, N, mask_ratio_tuple, mask_probability,
                                 mask_generator, bgswap_mask_generator):
    n_samples_masked = int(B * mask_probability)
    probs = torch.linspace(*mask_ratio_tuple, n_samples_masked + 1)
    upperbound = 0
    masks_list = []
    for i in range(0, n_samples_masked):
        prob_min = probs[i]
        prob_max = probs[i + 1]
        masks_list.append(
            torch.BoolTensor(
                mask_generator(int(N * random.uniform(prob_min, prob_max)))))
        upperbound += int(N * prob_max)
    for i in range(n_samples_masked, B):
        masks_list.append(torch.BoolTensor(mask_generator(0)))

    random.shuffle(masks_list)

    collated_masks = torch.stack(masks_list).flatten(1)
    mask_indices_list = collated_masks.flatten().nonzero().flatten()
    bgs_fg_masks = 1 - torch.stack([
        bgswap_mask_generator(return_flat_mask=True)
        for _ in range(len(masks_list))
    ])
    masks_weight = (1 / (collated_masks * bgs_fg_masks).sum(-1).clamp(min=1.0)
                    ).unsqueeze(-1).expand_as(collated_masks)[collated_masks]
    n_masked_patches = torch.full((1, ),
                                  fill_value=mask_indices_list.shape[0],
                                  dtype=torch.long)
    return {
        "collated_masks": collated_masks,
        "mask_indices_list": mask_indices_list,
        "masks_weight": masks_weight,
        "upperbound": upperbound,
        "n_masked_patches": n_masked_patches,
        "bgswap_masked_masks": bgs_fg_masks,
    }



In [ ]:
class CellCollator():

    def __init__(self,
                 mask_ratio_tuple,
                 mask_probability,
                 dtype,
                 omb_params,
                 omb_which="OuterBiasedMasker",
                 n_tokens=None,
                 mask_generator=None,
                 mask_outerbias_weight=False):

        self.mask_ratio_tuple = mask_ratio_tuple
        self.mask_probability = mask_probability
        self.dtype = dtype
        self.n_tokens = n_tokens
        self.mask_generator = mask_generator

        self.color_jittering = transforms.Compose(
            [  # for local views, deferred
                transforms.RandomApply(
                    [
                        transforms.ColorJitter(brightness=0.4,
                                               contrast=0.4,
                                               saturation=0.2,
                                               hue=0.1)
                    ],
                    p=0.8,
                ),
                transforms.RandomApply(
                    [GaussianNoise(min_var=0, max_var=0.05)], p=0.5),
                GaussianBlur(p=0.5, radius_max=1),
                transforms.RandomSolarize(threshold=0.5, p=.2),
                transforms.RandomGrayscale(p=0.2),
            ])
        #self.color_jittering = transforms.Compose([])

        if omb_which == "OuterBiasedMasker":
            self.obm = OuterBiasedMasker(**omb_params)
        elif omb_which == "OuterCircularMasker":
            self.obm = OuterCircularMasker(**omb_params)
        else:
            assert False

        self.mask_outerbias_weight = mask_outerbias_weight

    def __call__(self, samples_list):
        n_global_crops = len(samples_list[0][0]["global_crops"])
        n_local_crops = len(samples_list[0][0]["local_crops"])

        collated_global_crops = torch.stack([
            s[0]["global_crops"][i] for i in range(n_global_crops)
            for s in samples_list
        ])
        collated_local_crops = torch.stack([
            s[0]["local_crops"][i] for i in range(n_local_crops)
            for s in samples_list
        ])

        bg_swap_masks = torch.stack([
            self.obm() for _ in range(len(collated_local_crops))
        ]).unsqueeze(1)

        collated_local_crops = (
            collated_local_crops *
            (1 - bg_swap_masks) + collated_local_crops[torch.randperm(
                collated_local_crops.size(0))] * bg_swap_masks)
        collated_local_crops = torch.stack(
            [self.color_jittering(i) for i in collated_local_crops])

        images = {
            "collated_global_crops": collated_global_crops.to(self.dtype),
            "collated_local_crops": collated_local_crops.to(self.dtype),
            "bg_swap_masks": bg_swap_masks,
        }
        B = len(collated_global_crops)
        N = self.n_tokens

        if self.mask_outerbias_weight:
            images.update(
                create_mask_with_bgswap_mask(B, N, self.mask_ratio_tuple,
                                             self.mask_probability,
                                             self.mask_generator, self.obm))
        else:
            images.update(
                create_mask(
                    B,
                    N,
                    self.mask_ratio_tuple,
                    self.mask_probability,
                    self.mask_generator,
                ))

        return images

In [ ]:
def collate_data_and_cast(samples_list, mask_ratio_tuple, mask_probability, dtype, n_tokens=None, mask_generator=None):
    #print(len(samples_list))
    # dtype = torch.half  # TODO: Remove
    #torch.save(samples_list, "noaug_sample_list.pt")
    #exit(0)
    n_global_crops = len(samples_list[0][0]["global_crops"])
    n_local_crops = len(samples_list[0][0]["local_crops"])

    collated_global_crops = torch.stack([s[0]["global_crops"][i] for i in range(n_global_crops) for s in samples_list])

    if n_local_crops:
        collated_local_crops = torch.stack([s[0]["local_crops"][i] for i in range(n_local_crops) for s in samples_list])
    else:
        collated_local_crops = torch.empty(collated_global_crops.shape)

    B = len(collated_global_crops)
    N = n_tokens
    n_samples_masked = int(B * mask_probability)
    probs = torch.linspace(*mask_ratio_tuple, n_samples_masked + 1)
    upperbound = 0
    masks_list = []
    for i in range(0, n_samples_masked):
        prob_min = probs[i]
        prob_max = probs[i + 1]
        masks_list.append(torch.BoolTensor(mask_generator(int(N * random.uniform(prob_min, prob_max)))))
        upperbound += int(N * prob_max)
    for i in range(n_samples_masked, B):
        masks_list.append(torch.BoolTensor(mask_generator(0)))

    random.shuffle(masks_list)

    collated_masks = torch.stack(masks_list).flatten(1)
    mask_indices_list = collated_masks.flatten().nonzero().flatten()

    masks_weight = (1 / collated_masks.sum(-1).clamp(min=1.0)).unsqueeze(-1).expand_as(collated_masks)[collated_masks]

    return {
        "collated_global_crops": collated_global_crops.to(dtype),
        "collated_local_crops": collated_local_crops.to(dtype),
        "collated_masks": collated_masks,
        "mask_indices_list": mask_indices_list,
        "masks_weight": masks_weight,
        "upperbound": upperbound,
        "n_masked_patches": torch.full((1,), fill_value=mask_indices_list.shape[0], dtype=torch.long),
    }



# Main

## Vanilla DINOv2

In [ ]:
data = torch.load("out/noaug_sample_list.pt")
data = data[:50]

In [ ]:
rrc = CellAugmentationDINO()
sin = SRHInstanceNorm()
ccnad = CellCropNoAugmentationDINO()

img_size = 48 #cfg.crops.global_crops_size
patch_size = 4 #cfg.student.patch_size
n_tokens = (img_size // patch_size)**2
mask_generator = MaskingGenerator(
    input_size=(img_size // patch_size, img_size // patch_size),
    max_num_patches=0.5 * img_size // patch_size * img_size // patch_size,
)

dc = partial(collate_data_and_cast, mask_ratio_tuple=[0.1,0.5], mask_probability=0.5, dtype= torch.float, n_tokens=n_tokens, mask_generator=mask_generator)

In [ ]:
augmented = [(ccnad(d[0]), d[1]) for d in data]
collated = dc(augmented)

In [ ]:
for i in range(10):
    fig, ax = plt.subplots(1,9, figsize=(20,2))
    ax[0].imshow(einops.rearrange(srh_to_uint8_func(data[i][0]), "c h w -> h w c"))

    ax[1].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i]), "c h w -> h w c"))
    ax[2].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i+50]), "c h w -> h w c"))

    ax[3].imshow(collated["collated_masks"][i].reshape((12,12)))
    ax[4].imshow(collated["collated_masks"][i+50].reshape((12,12)))

    ax[5].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i]), "c h w -> h w c"))
    ax[6].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+50]), "c h w -> h w c"))
    ax[7].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+100]), "c h w -> h w c"))
    ax[8].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+150]), "c h w -> h w c"))

    for a_ in ax: a_.axis('off')
    plt.tight_layout()

    #pdf_pages.savefig(fig)
#pdf_pages.close()

In [ ]:
!mkdir /Users/chengjia/Desktop/dv2_masking

In [ ]:
for i in range(50):
    Image.fromarray(einops.rearrange(srh_to_uint8_func(data[i][0]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_orig64.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i]), "c h w -> h w c").numpy()).save(   f"/Users/chengjia/Desktop/dv2_masking/image{i}_g1.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i+50]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_g2.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+0]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local1.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+50]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local2.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+100]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local3.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+150]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local4.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+200]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local5.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+250]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local6.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+300]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local7.png")
    Image.fromarray(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+350]), "c h w -> h w c").numpy()).save(f"/Users/chengjia/Desktop/dv2_masking/image{i}_local8.png")

## Ours

In [ ]:
cc = CellCollator(
    mask_ratio_tuple=[0.1,0.5], #cfg.ibot.mask_ratio_min_max,
    mask_probability=0.5, #cfg.ibot.mask_sample_probability,
    n_tokens=n_tokens,
    mask_generator=mask_generator,
    dtype = torch.float, #inputs_dtype,
    omb_params={"mask_size":12,
                "token_size":4,
                "dist_power": 2,
                "sharpness_range":[0.2,10],
                "n_mask_token_perct_range":[0.5,0.75]
               },
    mask_outerbias_weight = True,
)


In [ ]:
augmented = [(rrc(d[0]), d[1]) for d in data]
collated = cc(augmented)


In [ ]:
#from matplotlib.backends.backend_pdf import PdfPages
#pdf_pages = PdfPages('ours_aug.pdf')

In [ ]:
for i in range(10):
    fig, ax = plt.subplots(1,13, figsize=(20,2))
    ax[0].imshow(einops.rearrange(srh_to_uint8_func(data[i][0]), "c h w -> h w c"))

    ax[1].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i]), "c h w -> h w c"))
    ax[2].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_global_crops"][i+50]), "c h w -> h w c"))

    #ax[3].imshow(collated["collated_masks"][i].reshape((12,12)))
    ax[3].imshow((collated["collated_masks"][i]).reshape((12,12)))
    ax[4].imshow(collated["collated_masks"][i+50].reshape((12,12)))

    ax[5].imshow((collated["bgswap_masked_masks"][i]).reshape((12,12)))
    ax[6].imshow((collated["bgswap_masked_masks"][i+50]).reshape((12,12)))
    
    
    ax[7].imshow((collated["collated_masks"][i] * collated["bgswap_masked_masks"][i]).reshape((12,12)))
    ax[8].imshow((collated["collated_masks"][i+50] * collated["bgswap_masked_masks"][i+50]).reshape((12,12)))
    

    
    ax[9].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i]), "c h w -> h w c"))
    ax[10].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+50]), "c h w -> h w c"))

    ax[11].imshow(collated["bg_swap_masks"][i].reshape((48,48)))
    ax[12].imshow(collated["bg_swap_masks"][i+50].reshape((48,48)))
    

    
    for a_ in ax: a_.axis('off')
    plt.tight_layout()

    #pdf_pages.savefig(fig)
#pdf_pages.close()

## Circular masker for eval

In [ ]:
import torch
import random
from typing import Sequence, Literal

class OuterCircularMasker:
    def __init__(
        self,
        mask_size: int,
        token_size: int,
        n_mask_token_perct_range: Sequence[float] = (.5, .75),
        mode: Literal["strict", "shell"] = "strict",
        outer_ring_width: int = 2,
        device: str = "cpu",
        dtype: torch.dtype = torch.float64,
    ):
        """
        Outer circular mask with two modes and an optional fixed outer square ring
        that is always masked.

        - outer_ring_width:
            Number of tokens from each border forming a square ring that is
            always masked (1). The circular masking is applied only to the
            remaining inner region.

        - mode="strict":
            Enforce the sampled masking fraction as closely as possible by
            randomly selecting within the boundary radius (within the inner region).
        - mode="shell":
            Keep all tokens at a given radius together (no splitting within a shell),
            so the actual masked fraction may deviate from the target.

        Args:
            mask_size: number of tokens per spatial side (mask_size x mask_size).
            token_size: upsampling factor per token (token_size x token_size).
            n_mask_token_perct_range: (min, max) fraction of tokens to mask
                within the inner region (excluding the fixed outer ring).
            mode: "strict" or "shell".
            outer_ring_width: number of tokens from border always masked (square ring).
            device: tensor device.
            dtype: dtype for distance computation.
        """
        self.mask_size = mask_size
        self.token_size = token_size
        self.device = device
        self.dtype = dtype
        self.frac_min, self.frac_max = n_mask_token_perct_range
        self.mode = mode
        self.outer_ring_width = int(outer_ring_width)

        if self.outer_ring_width < 0:
            raise ValueError("outer_ring_width must be >= 0")
        if self.outer_ring_width * 2 >= mask_size:
            raise ValueError(
                "outer_ring_width is too large: inner region would be empty "
                f"(outer_ring_width={self.outer_ring_width}, mask_size={mask_size})"
            )

        self.num_tokens = mask_size * mask_size

        # Coordinates and squared distances to center (for circular ordering).
        coords = torch.arange(mask_size, device=device, dtype=dtype)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")

        cx = (mask_size - 1) / 2.0
        cy = (mask_size - 1) / 2.0

        dist2 = (xx - cx) ** 2 + (yy - cy) ** 2  # [mask_size, mask_size]
        self.dist2_flat = dist2.reshape(-1)      # [num_tokens]

        # Define inner region (not part of the fixed outer ring).
        if self.outer_ring_width > 0:
            w = self.outer_ring_width
            inner = (
                (xx >= w) & (xx < mask_size - w) &
                (yy >= w) & (yy < mask_size - w)
            )
        else:
            inner = torch.ones_like(xx, dtype=torch.bool, device=device)

        inner_flat = inner.reshape(-1)
        self.interior_mask_flat = inner_flat                     # True for inner tokens
        self.interior_indices = torch.nonzero(
            inner_flat, as_tuple=False
        ).squeeze(1)                                             # [N_inner]
        self.ring_indices = torch.nonzero(
            ~inner_flat, as_tuple=False
        ).squeeze(1)                                             # [N_ring]
        self.num_tokens_inner = int(self.interior_indices.numel())

        # Distances restricted to inner region
        self.dist2_inner = self.dist2_flat[self.interior_indices]   # [N_inner]
        sorted_dist2_inner, order_inner = torch.sort(
            self.dist2_inner, descending=False, stable=True
        )
        self.sorted_dist2_inner = sorted_dist2_inner                # [N_inner]
        self.sorted_idx_inner = self.interior_indices[order_inner]  # [N_inner], global indices

        if mode == "strict" and self.num_tokens_inner > 0:
            # Shells over inner tokens only
            unique_vals, counts = torch.unique(
                self.sorted_dist2_inner, return_counts=True
            )
            start = torch.zeros_like(counts)
            start[1:] = torch.cumsum(counts[:-1], dim=0)

            self.shell_radii = unique_vals     # [S_inner]
            self.shell_counts = counts         # [S_inner]
            self.shell_start = start           # [S_inner]

        # Upsampling kernel for tokens -> pixels
        self.upsample_kernel = torch.ones(
            (token_size, token_size),
            dtype=torch.uint8,
            device=device,
        )

    def __call__(self) -> torch.Tensor:
        """
        Generate a mask where:
          - an outer square ring of width `outer_ring_width` is always masked (1),
          - the remaining inner region is masked according to a sampled fraction
            in [frac_min, frac_max] using a circular ordering.

        Returns:
            torch.Tensor of shape
            (mask_size * token_size, mask_size * token_size), dtype=torch.uint8.
        """
        # Start with all unmasked (0)
        mask_flat = torch.zeros(self.num_tokens, dtype=torch.uint8, device=self.device)

        # Always-mask outer ring (if any)
        if self.outer_ring_width > 0 and self.ring_indices.numel() > 0:
            mask_flat[self.ring_indices] = 1

        # If no inner tokens exist, we are done (whole map is outer ring)
        if self.num_tokens_inner == 0:
            base_mask = mask_flat.view(self.mask_size, self.mask_size)
            return torch.kron(base_mask, self.upsample_kernel)

        # Sample desired masked fraction over the INNER region only
        frac = random.uniform(self.frac_min, self.frac_max)
        num_masked_target_inner = int(round(frac * self.num_tokens_inner))
        num_masked_target_inner = max(0, min(self.num_tokens_inner, num_masked_target_inner))
        num_unmasked_target_inner = self.num_tokens_inner - num_masked_target_inner

        inner_idx = self.interior_indices  # global indices of inner tokens

        # Edge cases for inner region
        if num_unmasked_target_inner <= 0:
            # All inner tokens masked
            mask_flat[inner_idx] = 1
        elif num_unmasked_target_inner >= self.num_tokens_inner:
            # All inner tokens unmasked (outer ring already masked)
            # mask_flat[inner_idx] already 0, so nothing to do
            pass
        else:
            if self.mode == "shell":
                # Choose radius threshold so that all points with same radius are treated identically.
                k = num_unmasked_target_inner - 1
                thr_dist2 = self.sorted_dist2_inner[k]
                # Inner region: dist2 >= thr_dist2 -> masked (1), else unmasked (0)
                mask_inner = (self.dist2_inner >= thr_dist2).to(torch.uint8)
                mask_flat[inner_idx] = mask_inner

            elif self.mode == "strict":
                # Initialize inner region as fully masked
                mask_flat[inner_idx] = 1

                remaining = num_unmasked_target_inner
                unmasked_indices = []

                if self.num_tokens_inner > 0:
                    for shell_start, shell_count in zip(self.shell_start, self.shell_counts):
                        shell_start = int(shell_start.item())
                        shell_count = int(shell_count.item())

                        if remaining <= 0:
                            break

                        shell_idx = self.sorted_idx_inner[
                            shell_start : shell_start + shell_count
                        ]  # global indices for this shell

                        if remaining >= shell_count:
                            # Take the entire shell as unmasked
                            unmasked_indices.append(shell_idx)
                            remaining -= shell_count
                        else:
                            # Boundary shell: randomly select 'remaining' tokens in this shell
                            perm = torch.randperm(shell_count, device=self.device)[:remaining]
                            chosen = shell_idx[perm]
                            unmasked_indices.append(chosen)
                            remaining = 0
                            break

                if unmasked_indices:
                    unmasked_indices = torch.cat(unmasked_indices, dim=0)
                    mask_flat[unmasked_indices] = 0  # 0 = unmasked
            else:
                raise ValueError(f"Unknown mode: {self.mode}")

        base_mask = mask_flat.view(self.mask_size, self.mask_size)
        return torch.kron(base_mask, self.upsample_kernel)

In [ ]:

class SingleCellTokenMaskShuffleCollator:  # used for perturbation evaluations
    def __init__(self,
                 mask_generator_which,
                 mask_generator_params,
                 strong_transforms,
                 use_mean: bool = False):
        """
        Args:
            mask_generator: callable with signature () -> 2D mask (H, W)
                Typically an instance of OuterCircularMasker.
            strong_transforms: augmentation pipeline applied to blended images.
            use_mean (bool): if True, background is replaced by per-image mean.
        """
        if mask_generator_which=="OuterBiasedMasker":
            self.mask_generator = OuterBiasedMasker(**mask_generator_params)
        elif mask_generator_which=="OuterCircularMasker":
            self.mask_generator = OuterCircularMasker(**mask_generator_params)
        else:
            assert False

        self.transform = strong_transforms
        self.use_mean = use_mean


    def blend_batch(self, fg_im, bg_im, fg_mask):
        """
        fg_im: (B, C, H, W)
        bg_im: (B, C, H, W)
        fg_mask: (B, H, W), uint8/bool, 1=fg, 0=bg
        """
        fg_mask = fg_mask.to(device=fg_im.device, dtype=fg_im.dtype)
        fg_mask = fg_mask.unsqueeze(1)                        # (B, 1, H, W)
        fg_mask = fg_mask.expand(-1, fg_im.shape[1], -1, -1)  # (B, C, H, W)

        if self.use_mean:
            bg_mean = bg_im.mean(dim=[2,3], keepdim=True)     # (B, C, 1, 1)
            return fg_im * fg_mask + bg_mean * (1 - fg_mask)
        else:
            return fg_im * fg_mask + bg_im * (1 - fg_mask)

    def __call__(self, raw_batch):
        # all_images: (B, aug, C, H, W)
        all_images = torch.stack([i["image"] for i in raw_batch])
        assert all_images.shape[1] == 1  # one view per sample

        # Take first (and only) aug
        fg = torch.clone(all_images[:, 0, ...])  # (B, C, H, W)

        # Build shuffled background
        bg = torch.clone(all_images[:, 0, ...])  # (B, C, H, W)
        bg = bg[torch.randperm(len(bg))]

        B, C, H, W = fg.shape

        # Generate one mask per sample via the mask_generator
        # mask_generator() -> (H, W) with values in {0,1}
        masks = torch.stack(
            [1 - self.mask_generator().to(fg.device) for _ in range(B)],  # (B, H, W)
            dim=0,
        )

        # Blend foreground/background using the masks
        fg_blended = self.blend_batch(fg, bg, masks)  # (B, C, H, W)

        # Apply strong transforms (per-sample)
        # You were slicing [:3]; keep that if your transform expects up to 3 channels.
        fg_aug = torch.stack(
            [self.transform(img) for img in fg_blended[:, :3, ...]],
            dim=0,
        )

        return {
            "image": fg_aug.unsqueeze(1),  # (B, 1, C, H, W) to match original API
            "label": torch.stack([i["label"] for i in raw_batch]),
            "path": [[i["path"][0] for i in raw_batch]],
        }


In [ ]:
masks = []
for nmtpr in np.arange(0,1.1,0.1):
    mg = OuterCircularMasker(**{"mask_size":16,
                        "token_size":4,
                        "n_mask_token_perct_range":[nmtpr, nmtpr],
                        "mode":"strict",
                       })
    masks.append([mg()[8:-8,8:-8], mg()[8:-8,8:-8]])

In [ ]:
len(masks)

In [ ]:
C = len(masks)
R = len(masks[0])

fig, ax = plt.subplots(
    R, C,
    figsize=(C, R),
    squeeze=False,
    constrained_layout=True,
    dpi=300
)

for i in range(C):
    for j in range(R):
        m = masks[i][j]
        im = ax[j, i].imshow(m.cpu())
        ax[j, i].axis('off')
        im.set_clim(0, 1)

fig.savefig("circular_mask_viz.png", dpi=300, bbox_inches='tight', pad_inches=0)

In [ ]:
for nmtpr in np.arange(0,1.1,0.1):

    eval_collater = SingleCellTokenMaskShuffleCollator(
                 mask_generator_params={"mask_size":16,
                    "token_size":4,
                    "n_mask_token_perct_range":[nmtpr, nmtpr],
                    "mode":"strict",
                   },
        mask_generator_which="OuterCircularMasker",
                 strong_transforms=lambda x:x,
    use_mean=True,)
    
    augmented = [{"image": i[0].unsqueeze(0), "label": torch.tensor([0]), "path":[""]} for i in data]
    
    collated = eval_collater(augmented)

    for i in range(3,4):
        fig, ax = plt.subplots(1,9, figsize=(20,2))
        ax[0].imshow(einops.rearrange(srh_to_uint8_func(data[i][0]), "c h w -> h w c"))

        ax[1].imshow(einops.rearrange(srh_to_uint8_func(collated["image"][i]), "1 c h w -> h w c")[8:-8,8:-8,:])
        #ax[2].imshow(einops.rearrange(srh_to_uint8_func(collated["image"][i+50]), "1 c h w -> h w c"))

        #ax[3].imshow(collated["collated_masks"][i].reshape((12,12)))
        #ax[4].imshow(collated["collated_masks"][i+50].reshape((12,12)))

        #ax[5].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i]), "c h w -> h w c"))
        #ax[6].imshow(einops.rearrange(srh_to_uint8_func(collated["collated_local_crops"][i+50]), "c h w -> h w c"))

        #ax[7].imshow(collated["bg_swap_masks"][i].reshape((48,48)))
        #ax[8].imshow(collated["bg_swap_masks"][i+50].reshape((48,48)))

        #print(collated["bg_swap_masks"][i][8:-8,8:-8].sum() / (48*48))

        for a_ in ax: a_.axis('off')
        plt.tight_layout()

        #pdf_pages.savefig(fig)
    #pdf_pages.close()

In [ ]:
collated["image"].shape

In [ ]:
torch.stack([i["image"] for i in raw_batch])